# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible guide for loading, exploring, and processing the FAIR^2 mass spectrometry dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the latest mlcroissant library is installed
!pip install --upgrade mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using mlcroissant.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata and create the dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {getattr(metadata, 'name', '')}\n")
print(f"Description: {getattr(metadata, 'description', '')}\n")
# Show additional information
print(f"Published: {getattr(metadata, 'datePublished', '')}")
print(f"Identifier: {getattr(metadata, 'identifier', '')}")

## 2. Data Overview
This section reviews available record sets, fields, and their `@id`s in the dataset.

A "record set" in Croissant is a tabular data unit (e.g., a CSV or Excel-like table), and each record set consists of fields/columns, each identified by their `@id`.

**Let's enumerate record sets and fields by their `@id`s:**

In [ ]:
from pprint import pprint

# Gather all record set @ids and their fields
record_sets = getattr(metadata, 'recordSet', [])
if isinstance(record_sets, dict):
    record_sets = [record_sets]

if not record_sets:
    # mlcroissant will generate record sets from data files if not listed directly.
    # So let's enumerate them from dataset directly as a fallback.
    print("No explicit record sets in metadata; trying to discover from dataset.records().")
    # For demonstration, attempt to list up to 3 available record sets.
    # Use dataset._record_set_ids (non-public) in mlcroissant, or use knowledge about the data files.
    # Here, use the main package's known record set id:
    discovered_record_set_ids = [
        'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034/main'  # The main data table
    ]
else:
    discovered_record_set_ids = [getattr(rs, '@id', None) or rs for rs in record_sets]

print("Available record set @ids:")
for rsid in discovered_record_set_ids:
    print(f"  - {rsid}")
    # Try to get field ids for each record set
    try:
        fields = dataset.fields(record_set=rsid)
        print("    Fields (by @id):")
        for f in fields:
            print(f"      * {getattr(f, '@id', str(f))} ({f.name if hasattr(f, 'name') else ''})")
    except Exception as e:
        print(f"    [Could not enumerate fields: {e}]")

## 3. Data Extraction
Load data from a record set into a DataFrame. All fields and record sets are referenced by their `@id` as above. 

Below, we extract data from the discovered main record set.

In [ ]:
# List of record sets to extract; update if more discovered
record_sets = discovered_record_set_ids
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records.")
    print(f"Fields (DataFrame columns):\n{list(df.columns)}\n")

# For convenience, use the first (only) record set for further analysis
main_record_set = record_sets[0]
df = dataframes[main_record_set]

# Show a sample
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping. 

For this dataset, let’s:

- Select a numeric field (e.g., "Molecular Weight" or peptide count)
- Remove proteins with low abundance
- Normalize the selected numeric field
- Group by a categorical field, if available (e.g., protein type/category)

**All fields used are referenced by their `@id`.**

In [ ]:
# Example: Analyze 'Molecular Weight' and filter for high-abundance proteins

# Let's try to identify the correct column name for 'Molecular Weight' (MW), typically 'MW' or similar.
print("Field sample and types:")
print(df.dtypes)

# Use one of the known numeric fields. Replace with actual field name from print(df.columns) if different.
numeric_field = None
for col in df.columns:
    if col.lower() in ["mw", "molecular_weight", "molecularweight", "molecular weight"]:
        numeric_field = col
        break
if numeric_field is None:
    # Fallback: use any float column
    float_cols = df.select_dtypes(include=['float', 'float32', 'float64', 'int']).columns
    if len(float_cols) > 0:
        numeric_field = float_cols[0]
    else:
        raise ValueError("No numeric column found for EDA.")

print(f"Selected numeric field (column): {numeric_field}")

# Filter to only proteins with MW > threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered proteins with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records left.")

# Normalize the selected numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

print(f"Normalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by another field, e.g., a category/descriptor column
group_field = None
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
for col in categorical_cols:
    if 'category' in col.lower() or 'type' in col.lower() or 'description' in col.lower():
        group_field = col
        break
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nMean '{numeric_field}' grouped by '{group_field}':")
    print(grouped_df.head())
else:
    print("No suitable categorical group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For example: Distribution of Molecular Weight among abundant proteins.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the selected numeric field (e.g., Molecular Weight)
plt.figure(figsize=(8,4))
filtered_df[numeric_field].hist(bins=30)
plt.title(f"Distribution of {numeric_field} (filtered)")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Optional: If a group_field was found, boxplots
if group_field is not None:
    plt.figure(figsize=(12, 5))
    filtered_df.boxplot(column=numeric_field, by=group_field, rot=90)
    plt.title(f"{numeric_field} by {group_field}")
    plt.suptitle("")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library, referencing all entities by their `@id`s. We loaded metadata, explored available record sets and fields, extracted tabular data to Pandas, and performed basic exploratory analysis and visualization.

Further analyses could include protein annotation enrichment, advanced outlier detection, or linking protein IDs to external biological databases.
